# Master Workflow Script

this the master workflow script, run the following cells to prepare data for visualization

In [1]:
from modules.subsampler.core import *
import modules.get_probabilistic_forecast.core as prob
from pathlib import Path
from tqdm import tqdm
import gc
import xarray as xr
import pickle

## Step 1 Generate Probabilistic Forecast Data Using Hindcast

### Setup Directories 

In [2]:
# Variable definitions
list_of_variables = {
    'Rainf_tavg': 'Average precipitation', 
    'Qair_f_tavg': 'Specific humidity',
    'Qs_tavg': 'Surface runoff',
    'Evap_tavg': 'Evapotranspiration',
    'Tair_f_tavg': 'Avg. air temperature',
    'SoilMoist_inst': 'Soil moisture',
    'SoilTemp_inst': 'Soil temperature',
    'Streamflow_tavg': 'Stream flow'
}

# Data directory
# surface_model_file_path = r"/mnt/vast/prakrut/backup/malaria_amazon/amazon_forecast" # Input location on group server
surface_model_file_path = rf"C:\Users\Kris\Documents\amazonforecast\data\hindcast\amazon_forecast" # Input local on local machine [for test purposes]


# Find forecast and hindcast files
try: 
    forecast_file, hindcast_files, f_dt = prob.split_forecast_and_dec_hindcasts(surface_model_file_path)
    print("Forecast file:", forecast_file)
    print("Hindcasts   :", len(hindcast_files), "files")
    print("Init date   :", f_dt)
    # Create initialization date tag
    initialization_date = f"{f_dt.year}_{f_dt.strftime('%b').lower()}"
    print("Forecast initialization date:", initialization_date)

except Exception as e:
    print(f"{type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()


# Create output directories
output_dir = Path('./get_ldas_probabilistic_output')
output_dir.mkdir(exist_ok=True, parents=True)

# Create output directories for nc files
output_dir_cache = Path('./get_ldas_probabilistic_output/tmp')
output_dir_cache.mkdir(exist_ok=True, parents=True)

subsampled_output_dir = output_dir / 'subsampled'
subsampled_output_dir.mkdir(exist_ok=True, parents=True)

# for var in list_of_variables.keys(): # [NOTE: these lines are obsolete]
#     geojson_output_dir_per_var = geojson_output_dir / var
#     geojson_output_dir_per_var.mkdir(exist_ok=True, parents=True)

print(f"\n Output directory: {output_dir}")
print(f"Subsampled directory: {subsampled_output_dir} \n")

Forecast file: C:\Users\Kris\Documents\amazonforecast\data\hindcast\amazon_forecast\ldas_fcst_2024_dec01.nc
Hindcasts   : 23 files
Init date   : 2024-12-01 00:00:00
Forecast initialization date: 2024_dec

 Output directory: get_ldas_probabilistic_output
Subsampled directory: get_ldas_probabilistic_output\subsampled 



In [3]:
hindcast = prob.read_trim_hindcast(hindcast_files, 'Rainf_tavg')
forecast = prob.read_trim_forecast(forecast_file, 'Rainf_tavg')

In [4]:
probs = prob.calculate_probabilities(hindcast, forecast) * 100

Computing probabilities...
  category=0
  category=1
  category=2


### Main processing loop

In [4]:
# Process each variable
for variable, variable_longname in tqdm(list_of_variables.items()):  # Fixed: .items()
    print(f"\n{'='*60}")
    print(f"{variable_longname} ({variable})")
    print('='*60)
    
    try:
        # Load data
        print("Loading hindcast data...")
        hindcast = prob.read_trim_hindcast(hindcast_files, variable)
        print(f"  Shape: {hindcast.shape}")
        
        print("Loading forecast data...")
        forecast = prob.read_trim_forecast(forecast_file, variable)
        print(f"  Shape: {forecast.shape}")
        
        # Calculate probabilities (convert to percentages)
        print("Calculating tercile probabilities...")
        probs = prob.calculate_probabilities(hindcast, forecast) * 100
        print(f"\nProbability data shape: {probs.shape}")
        print(f"Dimensions: {probs.dims} Categories: {probs.category.values}")
        #print(f"Time steps: {len(probs.time)}")
        
        # Keep only maximum probability per category
        print("Filtering for maximum probabilities...")
        probs_with_nan = probs.where(probs == probs.max(dim='category'))
        
        # Determine output file path base
        file_base = output_dir_cache / f'prob_{initialization_date}_tercile_probability_max_{variable}'
        
        # Save by profile level for soil variables
        if variable in ['SoilMoist_inst', 'SoilTemp_inst']:
            print(f"\nProcessing soil variable with profile levels...")
            
            # Find profile dimension (various possible names)
            profile_dims = [d for d in probs_with_nan.dims 
                           if 'profile' in d.lower() or d in ['level', 'depth', 'SoilMoist_profiles']]
            
            if profile_dims:
                profile_dim = profile_dims[0]
                n_levels = len(probs_with_nan[profile_dim])
                print(f"  Found {n_levels} levels in dimension: '{profile_dim}'")
                print(f"  Level values: {probs_with_nan[profile_dim].values}")
                
                # Save each level separately
                for level_idx in range(n_levels):
                    level_data = probs_with_nan.isel({profile_dim: level_idx})
                    output_file = f'{file_base}_lvl_{level_idx}.nc'
                    level_data.to_netcdf(output_file)
                    print(f"  ✓ Saved level {level_idx} → {Path(output_file).name}")
            else:
                print(f"  ⚠ WARNING: No profile dimension found")
                print(f"    Available dimensions: {list(probs_with_nan.dims)}")
                print(f"    Saving as single file (lvl_0)")
                probs_with_nan.to_netcdf(f'{file_base}_lvl_0.nc')
    
        elif variable == 'Streamflow_tavg': # Extract river network
            river_mask_file = Path(f'./static/annual_mean_50cumecs_river_network.nc')
            if river_mask_file.exists():
                river_network_ds = xr.open_dataset(river_mask_file)
                river_mask = river_network_ds['mask']
                print(f"\n{'='*60}")
                print("Loaded river mask: \n")
                print(river_mask)
                print(f"\n{'='*60}")
            else: 
                print(f"File not found: {river_mask_file}")
            probs_with_nan = probs_with_nan.where(river_mask)
            # Non-soil variables: save as level 0
            output_file = f'{file_base}_lvl_0.nc'
            probs_with_nan.to_netcdf(output_file)
            print(f"  ✓ Saved → {Path(output_file).name}")
            
        else:
            # Non-soil variables: save as level 0
            output_file = f'{file_base}_lvl_0.nc'
            probs_with_nan.to_netcdf(output_file)
            print(f"  ✓ Saved → {Path(output_file).name}")
        
        print(f"\n✓ Completed {variable}")
        
    except Exception as e:
        print(f"\n✗ ERROR processing {variable}:")
        print(f"  {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        continue
    
    finally:
        # Clean up memory
        print("Cleaning up memory...")
        try:
            del hindcast, forecast, probs, probs_with_nan
        except:
            pass
        gc.collect()

print("\n" + "="*60)
print("✓ All variables processed!")
print("="*60)

  0%|          | 0/8 [00:00<?, ?it/s]


Average precipitation (Rainf_tavg)
Loading hindcast data...
  Shape: (138, 7, 540, 660)
Loading forecast data...
  Shape: (6, 7, 540, 660)
Calculating tercile probabilities...
Computing probabilities...
  category=0
  category=1
  category=2

Probability data shape: (3, 6, 540, 660)
Dimensions: ('category', 'time', 'lat', 'lon') Categories: [0 1 2]
Filtering for maximum probabilities...


 12%|█▎        | 1/8 [00:11<01:23, 11.86s/it]

  ✓ Saved → prob_2024_dec_tercile_probability_max_Rainf_tavg_lvl_0.nc

✓ Completed Rainf_tavg
Cleaning up memory...

Specific humidity (Qair_f_tavg)
Loading hindcast data...
  Shape: (138, 7, 540, 660)
Loading forecast data...
  Shape: (6, 7, 540, 660)
Calculating tercile probabilities...
Computing probabilities...
  category=0
  category=1
  category=2

Probability data shape: (3, 6, 540, 660)
Dimensions: ('category', 'time', 'lat', 'lon') Categories: [0 1 2]
Filtering for maximum probabilities...


 25%|██▌       | 2/8 [00:18<00:51,  8.53s/it]

...

  ✓ Saved level 3 → prob_2024_dec_tercile_probability_max_SoilMoist_inst_lvl_3.nc

✓ Completed SoilMoist_inst
Cleaning up memory...

Soil temperature (SoilTemp_inst)
Loading hindcast data...
  Shape: (138, 4, 7, 540, 660)
Loading forecast data...
  Shape: (6, 4, 7, 540, 660)
Calculating tercile probabilities...
Computing probabilities...
  category=0
  category=1
  category=2

Probability data shape: (3, 6, 4, 540, 660)
Dimensions: ('category', 'time', 'SoilTemp_profiles', 'lat', 'lon') Categories: [0 1 2]
Filtering for maximum probabilities...

Processing soil variable with profile levels...
  Found 4 levels in dimension: 'SoilTemp_profiles'
  Level values: [0 1 2 3]
  ✓ Saved level 0 → prob_2024_dec_tercile_probability_max_SoilTemp_inst_lvl_0.nc
  ✓ Saved level 1 → prob_2024_dec_tercile_probability_max_SoilTemp_inst_lvl_1.nc
  ✓ Saved level 2 → prob_2024_dec_tercile_probability_max_SoilTemp_inst_lvl_2.nc


 88%|████████▊ | 7/8 [02:19<00:31, 31.29s/it]

  ✓ Saved level 3 → prob_2024_dec_tercile_probability_max_SoilTemp_inst_lvl_3.nc

✓ Completed SoilTemp_inst
Cleaning up memory...

Stream flow (Streamflow_tavg)
Loading hindcast data...
  Shape: (138, 7, 540, 660)
Loading forecast data...
  Shape: (6, 7, 540, 660)
Calculating tercile probabilities...
Computing probabilities...
  category=0
  category=1
  category=2

Probability data shape: (3, 6, 540, 660)
Dimensions: ('category', 'time', 'lat', 'lon') Categories: [0 1 2]
Filtering for maximum probabilities...

Loaded river mask: 

<xarray.DataArray 'mask' (lat: 540, lon: 660)> Size: 3MB
[356400 values with dtype=int64]
Coordinates:
  * lat      (lat) float64 4kB -20.98 -20.93 -20.88 -20.82 ... 5.875 5.925 5.975
  * lon      (lon) float64 5kB -81.97 -81.92 -81.88 ... -49.13 -49.08 -49.03



100%|██████████| 8/8 [02:26<00:00, 18.33s/it]

  ✓ Saved → prob_2024_dec_tercile_probability_max_Streamflow_tavg_lvl_0.nc

✓ Completed Streamflow_tavg
Cleaning up memory...

✓ All variables processed!


## Step 2 Apply Sub-sampler for Web Usage

In [9]:
# Data bounds for the region
data_bounds = {'lon_min': -81.975, 'lon_max': -49.025, 'lat_min': -20.975, 'lat_max': 5.975}

# Get all probability netCDF files from the cache directory
prob_files = list(output_dir_cache.glob('prob_*.nc'))
print(f"Found {len(prob_files)} files to process\n")

Found 14 files to process



In [11]:
for nc_file in tqdm(prob_files):
    print(f"\n{'='*60}")
    print(f"Processing: {nc_file.name}")
    print('='*60)
    
    try:
        # Load the data
        ds = xr.open_dataarray(nc_file)
        print(f"  Shape: {ds.shape}")
        print(f"  Dims: {ds.dims}")

        # Create subsampler and generate pyramid
        subsampled = HydroViewerSubsampler(ds, resolution=256)
        pyramid, grain_map = subsampled.generate_pyramid(zooms=[4, 5, 6, 7, 8, 9])

        # Package result
        result = {
            'pyramid': pyramid, 
            'grain_map': grain_map, 
            'data_bounds': data_bounds
        }
        
        # Save with matching name
        output_name = nc_file.stem + '_subsampled.pkl'
        output_path = subsampled_output_dir / output_name
        
        with open(output_path, 'wb') as f:
            pickle.dump(result, f)
        
        print(f"\n  ✓ Saved → {output_name}")

    except Exception as e:
        print(f"\n  ✗ ERROR: {type(e).__name__}: {e}")
        continue

print(f"\n{'='*60}")
print("✓ All files processed!")
print(f"Output directory: {subsampled_output_dir}")
print('='*60)

  0%|          | 0/14 [00:00<?, ?it/s]


Processing: prob_2024_dec_tercile_probability_max_Evap_tavg_lvl_0.nc
  Shape: (3, 6, 540, 660)
  Dims: ('category', 'time', 'lat', 'lon')
Full grid: 540×660 = 356,400 cells
Grid spacing: Δlat=0.0500°, Δlon=0.0500°

GENERATING RESOLUTION PYRAMID

Available grain sizes: [ 1  2  3  4  5  6 10 12 15 20 30 60]
Available resolutions (m): [  5550.          11100.          16650.          22200.
  27750.          33300.          55500.          66600.
  83250.         110999.99999999 166499.99999999 332999.99999998]
Zoom 4: target=9773m, actual=11100m, grain=2
Zoom 5: target=4886m, actual=5550m, grain=1
Zoom 6: target=2443m, actual=5550m, grain=1
Zoom 7: target=1222m, actual=5550m, grain=1
Zoom 8: target=611m, actual=5550m, grain=1
Zoom 9: target=305m, actual=5550m, grain=1

Unique grain sizes needed: [1, 2]

Subsampling data...
  Subsampling with grain 2 (NaN-aware)...


  7%|▋         | 1/14 [00:52<11:28, 52.97s/it]

  Grain 2: 270×330 = 89,100 cells (75.0% reduction)
NaN fraction: 45.8% (land/ocean mask)

PYRAMID SUMMARY
Zoom 4:  270× 330 cells, grain= 2
Zoom 5:  540× 660 cells, grain= 1
Zoom 6:  540× 660 cells, grain= 1
Zoom 7:  540× 660 cells, grain= 1
Zoom 8:  540× 660 cells, grain= 1
Zoom 9:  540× 660 cells, grain= 1

  ✓ Saved → prob_2024_dec_tercile_probability_max_Evap_tavg_lvl_0_subsampled.pkl

Processing: prob_2024_dec_tercile_probability_max_Qair_f_tavg_lvl_0.nc
  Shape: (3, 6, 540, 660)
  Dims: ('category', 'time', 'lat', 'lon')
Full grid: 540×660 = 356,400 cells
Grid spacing: Δlat=0.0500°, Δlon=0.0500°

GENERATING RESOLUTION PYRAMID

Available grain sizes: [ 1  2  3  4  5  6 10 12 15 20 30 60]
Available resolutions (m): [  5550.          11100.          16650.          22200.
  27750.          33300.          55500.          66600.
  83250.         110999.99999999 166499.99999999 332999.99999998]
Zoom 4: target=9773m, actual=11100m, grain=2
Zoom 5: target=4886m, actual=5550m, grain=1
Z

 14%|█▍        | 2/14 [01:50<11:07, 55.60s/it]

...

  Grain 2: 270×330 = 89,100 cells (75.0% reduction)
NaN fraction: 77.1% (land/ocean mask)

PYRAMID SUMMARY
Zoom 4:  270× 330 cells, grain= 2
Zoom 5:  540× 660 cells, grain= 1
Zoom 6:  540× 660 cells, grain= 1
Zoom 7:  540× 660 cells, grain= 1
Zoom 8:  540× 660 cells, grain= 1
Zoom 9:  540× 660 cells, grain= 1

  ✓ Saved → prob_2024_dec_tercile_probability_max_SoilTemp_inst_lvl_3_subsampled.pkl

Processing: prob_2024_dec_tercile_probability_max_Streamflow_tavg_lvl_0.nc
  Shape: (3, 6, 540, 660)
  Dims: ('category', 'time', 'lat', 'lon')
Full grid: 540×660 = 356,400 cells
Grid spacing: Δlat=0.0500°, Δlon=0.0500°

GENERATING RESOLUTION PYRAMID

Available grain sizes: [ 1  2  3  4  5  6 10 12 15 20 30 60]
Available resolutions (m): [  5550.          11100.          16650.          22200.
  27750.          33300.          55500.          66600.
  83250.         110999.99999999 166499.99999999 332999.99999998]
Zoom 4: target=9773m, actual=11100m, grain=2
Zoom 5: target=4886m, actual=5550m, g

 93%|█████████▎| 13/14 [15:08<01:13, 73.03s/it]

  Grain 2: 270×330 = 89,100 cells (75.0% reduction)
NaN fraction: 92.0% (land/ocean mask)

PYRAMID SUMMARY
Zoom 4:  270× 330 cells, grain= 2
Zoom 5:  540× 660 cells, grain= 1
Zoom 6:  540× 660 cells, grain= 1
Zoom 7:  540× 660 cells, grain= 1
Zoom 8:  540× 660 cells, grain= 1
Zoom 9:  540× 660 cells, grain= 1

  ✓ Saved → prob_2024_dec_tercile_probability_max_Streamflow_tavg_lvl_0_subsampled.pkl

Processing: prob_2024_dec_tercile_probability_max_Tair_f_tavg_lvl_0.nc
  Shape: (3, 6, 540, 660)
  Dims: ('category', 'time', 'lat', 'lon')
Full grid: 540×660 = 356,400 cells
Grid spacing: Δlat=0.0500°, Δlon=0.0500°

GENERATING RESOLUTION PYRAMID

Available grain sizes: [ 1  2  3  4  5  6 10 12 15 20 30 60]
Available resolutions (m): [  5550.          11100.          16650.          22200.
  27750.          33300.          55500.          66600.
  83250.         110999.99999999 166499.99999999 332999.99999998]
Zoom 4: target=9773m, actual=11100m, grain=2
Zoom 5: target=4886m, actual=5550m, gra

100%|██████████| 14/14 [16:24<00:00, 70.30s/it]

  Grain 2: 270×330 = 89,100 cells (75.0% reduction)
NaN fraction: 81.2% (land/ocean mask)

PYRAMID SUMMARY
Zoom 4:  270× 330 cells, grain= 2
Zoom 5:  540× 660 cells, grain= 1
Zoom 6:  540× 660 cells, grain= 1
Zoom 7:  540× 660 cells, grain= 1
Zoom 8:  540× 660 cells, grain= 1
Zoom 9:  540× 660 cells, grain= 1

  ✓ Saved → prob_2024_dec_tercile_probability_max_Tair_f_tavg_lvl_0_subsampled.pkl

✓ All files processed!
Output directory: get_ldas_probabilistic_output\subsampled


In [8]:
# Process each variable
for variable, variable_longname in tqdm(list_of_variables.items()):  # Fixed: .items()
    print(f"\n{'='*60}")
    print(f"{variable_longname} ({variable})")
    ds = xr.open_dataset()
    ds = ds[f'{variable}']
    subsampled = HydroViewerSubsampler(ds, resolution=256)
    # Generate pyramid with zoom levels 4-9 (so zoom 4 = coarsest grain)
    pyramid, grain_map = subsampled.generate_pyramid(zooms=[4, 5, 6, 7, 8, 9])
    pyramid = {'pyramid': pyramid, 'grain_map': grain_map, 'data_bounds':data_bounds}

    pickle.dump(pyramid, open(f"./get_ldas_probabbilisitc_output/subsampled/prob_{initialization_date}_subsampled_tercile_prob_max_Rainf_tavg_lvl_0.pkl", "wb"))
    print('='*60)

100%|██████████| 8/8 [00:00<?, ?it/s]


Average precipitation (Rainf_tavg)

Specific humidity (Qair_f_tavg)

Surface runoff (Qs_tavg)

Evapotranspiration (Evap_tavg)

Avg. air temperature (Tair_f_tavg)

Soil moisture (SoilMoist_inst)

Soil temperature (SoilTemp_inst)

Stream flow (Streamflow_tavg)
